In [3]:
from pathlib import Path
import json
import pandas as pd
import torch
import ultralytics

# Move one directory up from notebooks/
PROJECT_ROOT = Path.cwd().parent

LABEL_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bdd100k_labels_extracted"
    / "100k"
)

TRAIN_LABELS = LABEL_ROOT / "train"
VAL_LABELS = LABEL_ROOT / "val"

print("Project root:", PROJECT_ROOT)
print("Label directory:", LABEL_ROOT)

print("\nDirectory checks:")
print("Label directory exists:", LABEL_ROOT.exists())
print("Train labels exist:", TRAIN_LABELS.exists())
print("Validation labels exist:", VAL_LABELS.exists())

if LABEL_ROOT.exists():
    print("\nDataset files:")
    print("Train JSON files:", len(list(TRAIN_LABELS.glob("*.json"))))
    print("Validation JSON files:", len(list(VAL_LABELS.glob("*.json"))))

print("\nEnvironment:")
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project root: c:\CH\NightSight
Label directory: c:\CH\NightSight\data\raw\bdd100k_labels_extracted\100k

Directory checks:
Label directory exists: True
Train labels exist: True
Validation labels exist: True

Dataset files:
Train JSON files: 70000
Validation JSON files: 10000

Environment:
PyTorch: 2.11.0+cu128
Ultralytics: 8.4.216
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [4]:
from collections import Counter

def inspect_split(label_dir):
    timeofday_counts = Counter()
    category_counts = Counter()
    total_images = 0

    for json_path in label_dir.glob("*.json"):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        total_images += 1

        attributes = data.get("attributes", {})
        timeofday = attributes.get("timeofday", "undefined")
        timeofday_counts[timeofday] += 1

        for frame in data.get("frames", []):
            for obj in frame.get("objects", []):
                category_counts[obj.get("category", "unknown")] += 1

    return total_images, timeofday_counts, category_counts


for split_name, split_dir in [
    ("TRAIN", TRAIN_LABELS),
    ("VALIDATION", VAL_LABELS)
]:
    total, times, categories = inspect_split(split_dir)

    print(f"\n{'=' * 50}")
    print(split_name)
    print(f"{'=' * 50}")

    print("Images:", total)
    print("\nTime-of-day distribution:")
    print(pd.Series(times).to_string())

    print("\nObject categories:")
    print(pd.Series(categories).sort_values(ascending=False).to_string())


TRAIN
Images: 70000

Time-of-day distribution:
daytime      36800
dawn/dusk     5033
night        28028
undefined      139

Object categories:
car                   714121
lane/single white     247108
traffic sign          239961
traffic light         186301
lane/road curb        109868
lane/crosswalk        108284
person                 91435
area/drivable          64050
area/alternative       61799
lane/double yellow     37519
truck                  30012
lane/single yellow     20220
bus                    11688
bike                    7227
lane/double white       5674
rider                   4522
motor                   3002
lane/single other        249
train                    136
lane/double other         26
area/unknown               2

VALIDATION
Images: 10000

Time-of-day distribution:
daytime      5258
night        3929
dawn/dusk     778
undefined      35

Object categories:
car                   102540
lane/single white      35452
traffic sign           34915
traffic light  

In [5]:
from collections import Counter
import json
import pandas as pd

TARGET_CLASSES = ["car", "person", "bus"]

def get_target_statistics(label_dir):
    time_counts = Counter()
    class_counts = Counter()
    class_images = Counter()

    for json_path in label_dir.glob("*.json"):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        timeofday = data.get("attributes", {}).get(
            "timeofday", "undefined"
        )
        time_counts[timeofday] += 1

        image_classes = set()

        for frame in data.get("frames", []):
            for obj in frame.get("objects", []):
                category = obj.get("category")

                if category in TARGET_CLASSES:
                    class_counts[category] += 1
                    image_classes.add(category)

        for category in image_classes:
            class_images[category] += 1

    return time_counts, class_counts, class_images


train_stats = get_target_statistics(TRAIN_LABELS)
val_stats = get_target_statistics(VAL_LABELS)

for name, stats in [
    ("TRAIN", train_stats),
    ("VALIDATION", val_stats)
]:
    times, objects, images = stats

    print(f"\n{'=' * 45}")
    print(name)
    print("=" * 45)

    print("\nTime-of-day:")
    display(pd.DataFrame(
        times.items(), columns=["Time of day", "Images"]
    ))

    print("\nTarget object instances:")
    display(pd.DataFrame([
        {"Class": c, "Instances": objects[c],
         "Images containing class": images[c]}
        for c in TARGET_CLASSES
    ]))


TRAIN

Time-of-day:


,Time of day,Images
0,daytime,36800
1,dawn/dusk,5033
2,night,28028
3,undefined,139



Target object instances:


,Class,Instances,Images containing class
0,car,714121,69201
1,person,91435,22104
2,bus,11688,9003



VALIDATION

Time-of-day:


,Time of day,Images
0,daytime,5258
1,night,3929
2,dawn/dusk,778
3,undefined,35



Target object instances:


,Class,Instances,Images containing class
0,car,102540,9879
1,person,13265,3220
2,bus,1597,1242


In [6]:
from collections import Counter, defaultdict
import json
import pandas as pd

TARGET_CLASSES = ["car", "person", "bus"]

def analyze_time_class_distribution(label_dir):

    image_counts = Counter()
    instance_counts = defaultdict(Counter)
    class_image_counts = defaultdict(Counter)

    for json_path in label_dir.glob("*.json"):

        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        timeofday = data.get("attributes", {}).get(
            "timeofday", "undefined"
        )

        image_counts[timeofday] += 1

        image_classes = Counter()

        for frame in data.get("frames", []):
            for obj in frame.get("objects", []):

                category = obj.get("category")

                if category in TARGET_CLASSES:
                    instance_counts[timeofday][category] += 1
                    image_classes[category] += 1

        for category in image_classes:
            class_image_counts[timeofday][category] += 1

    rows = []

    for timeofday in ["daytime", "night", "dawn/dusk"]:

        for category in TARGET_CLASSES:

            rows.append({
                "Time of day": timeofday,
                "Class": category,
                "Images": class_image_counts[timeofday][category],
                "Instances": instance_counts[timeofday][category]
            })

    return pd.DataFrame(rows), image_counts


train_distribution, train_time_counts = (
    analyze_time_class_distribution(TRAIN_LABELS)
)

val_distribution, val_time_counts = (
    analyze_time_class_distribution(VAL_LABELS)
)

print("TRAIN DISTRIBUTION")
display(train_distribution)

print("\nVALIDATION DISTRIBUTION")
display(val_distribution)

TRAIN DISTRIBUTION


,Time of day,Class,Images,Instances
0,daytime,car,36455,401754
1,daytime,person,14830,66109
2,daytime,bus,6344,8423
3,night,car,27632,255525
4,night,person,5594,18926
5,night,bus,1870,2229
6,dawn/dusk,car,4980,56034
7,dawn/dusk,person,1653,6328
8,dawn/dusk,bus,774,1015



VALIDATION DISTRIBUTION


,Time of day,Class,Images,Instances
0,daytime,car,5206,57852
1,daytime,person,2133,9368
2,daytime,bus,865,1156
3,night,car,3865,35941
4,night,person,837,2844
5,night,bus,263,302
6,dawn/dusk,car,775,8591
7,dawn/dusk,person,247,1046
8,dawn/dusk,bus,111,134


In [7]:
from collections import Counter
from itertools import combinations
import json
import pandas as pd

def analyze_class_cooccurrence(label_dir, target_time="night"):

    combinations_count = Counter()

    for json_path in label_dir.glob("*.json"):

        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        timeofday = data.get("attributes", {}).get(
            "timeofday", "undefined"
        )

        if timeofday != target_time:
            continue

        classes_present = set()

        for frame in data.get("frames", []):
            for obj in frame.get("objects", []):
                category = obj.get("category")

                if category in ["car", "person", "bus"]:
                    classes_present.add(category)

        if classes_present:
            key = " + ".join(sorted(classes_present))
            combinations_count[key] += 1

    return pd.DataFrame(
        [
            {"Class combination": key, "Images": count}
            for key, count in combinations_count.items()
        ]
    ).sort_values("Images", ascending=False)


print("NIGHTTIME TRAIN CLASS CO-OCCURRENCE")

display(
    analyze_class_cooccurrence(TRAIN_LABELS)
)

print("\nNIGHTTIME VALIDATION CLASS CO-OCCURRENCE")

display(
    analyze_class_cooccurrence(VAL_LABELS)
)

NIGHTTIME TRAIN CLASS CO-OCCURRENCE


,Class combination,Images
2,car,20833
1,car + person,4942
0,bus + car,1260
3,bus + car + person,597
4,person,48
5,bus + person,7
6,bus,6



NIGHTTIME VALIDATION CLASS CO-OCCURRENCE


,Class combination,Images
0,car,2857
1,car + person,749
2,bus + car,177
3,bus + car + person,82
5,person,5
4,bus,3
6,bus + person,1
